# ORM 学习文档 — 从本项目代码出发

本文档基于项目实际代码，逐层讲解 ORM（对象关系映射）的使用方式。

## 什么是 ORM？

ORM = **O**bject **R**elational **M**apping（对象关系映射）。

它的核心思想是：**用 Python 类来表示数据库表，用类的实例来表示表中的一行数据**。

没有 ORM 时，你需要手写 SQL：
```sql
INSERT INTO user (name, password, email) VALUES ('张三', 'xxx', 'zhangsan@example.com');
SELECT * FROM user WHERE name = '张三';
```

有了 ORM 后，你用 Python 对象操作就行：
```python
user = User(name="张三", password="xxx", email="zhangsan@example.com")
session.add(user)
session.commit()

user = session.exec(select(User).where(User.name == "张三")).first()
```

**好处：**
- 不用拼接 SQL 字符串，减少 SQL 注入风险
- 用 Python 类型系统做校验，编译期就能发现类型错误
- 换数据库（比如 MySQL → PostgreSQL）只需要改连接字符串，业务代码不用动

## 本项目用的是什么 ORM？

本项目使用 **SQLModel**，它是 SQLAlchemy（Python 最成熟的 ORM）和 Pydantic（数据校验库）的结合体。

- `SQLModel, table=True` → 定义数据库表（同时具备 Pydantic 的数据校验能力）
- `BaseModel`（纯 Pydantic）→ 定义请求体 / 响应体 / DTO，不涉及数据库

简单记忆：**需要建表就 `table=True`，不需要建表就 `BaseModel`**。

# 项目结构总览

```
app/
├── main.py                    # FastAPI app 实例、启动时建表、挂载路由
├── option.yml                 # jmcomic 下载配置
├── core/
│   ├── __init__.py
│   ├── database.py            # 数据库引擎、连接池、session 生成器
│   ├── redis.py               # Redis 客户端配置
│   ├── cache.py               # 缓存后端抽象（Protocol + MemoryCache + RedisCache）
│   └── lock.py                # 读写锁（RWLock）+ 按 key 粒度锁（KeyLock）
├── models/
│   ├── __init__.py
│   ├── tb_user.py             # SQLModel 数据模型（ORM 表定义）
│   ├── schemas.py             # 内部 DTO（Data Transfer Object），如 UserPublic
│   ├── request.py             # 请求体 Pydantic 模型
│   └── response.py            # 响应体 Pydantic 模型
├── repositories/
│   ├── __init__.py
│   └── user.py                # 用户数据访问层（所有 ORM 操作 + 密码工具）
├── routers/
│   ├── __init__.py
│   ├── user.py                # 用户路由
│   └── comic.py               # 漫画下载路由
└── services/
    ├── __init__.py
    ├── user.py                # 用户业务逻辑（调用 repo 层）
    └── comic.py               # 漫画下载业务逻辑（缓存 + double-check locking）
```

## 分层架构（4 层）

| 层 | 文件 | 职责 |
|---|---|---|
| **Router** | `routers/*.py` | 接收 HTTP 请求，调用 service，返回 HTTP 响应。不包含任何业务逻辑。 |
| **Service** | `services/*.py` | 业务逻辑编排。调用 repo 层获取/修改数据，组合业务规则，不直接操作 ORM。 |
| **Repository** | `repositories/*.py` | 数据访问层。封装所有 ORM 操作（CRUD）和密码哈希。返回 `UserPublic` 等不含敏感字段的 DTO。 |
| **Model** | `models/*.py` | 定义数据结构（数据库表、DTO、请求体、响应体）。 |

### 为什么要抽出 Repository 层？

旧架构中 service 层既做业务判断又做数据库操作，职责混杂。抽出 repo 层后：
- **Service** 只关心业务规则（"邮箱重复怎么办？"、"密码错误怎么提示？"）
- **Repository** 只关心数据存取（"怎么查？怎么插？密码怎么哈希？"）
- Service 层不再接触 `User` ORM 对象，而是通过 `UserPublic` 这种 DTO 来传递数据
- 密码哈希等实现细节被封装在 repo 内部，service 完全无感

# 第一步：数据库配置 — `app/core/database.py`

这个文件负责三件事：
1. 创建数据库引擎（Engine）
2. 根据模型自动建表
3. 提供数据库会话（Session）给其他层使用

In [ ]:
# app/core/database.py — 完整代码 + 逐行注释

import os

from dotenv import load_dotenv     # 从 .env 文件读取环境变量
from sqlmodel import SQLModel, Session, create_engine

# --------------------------------------------------
# 1. 加载环境变量
# --------------------------------------------------
# load_dotenv() 会读取项目根目录的 .env 文件，
# 把里面的键值对注入到 os.environ 中。
# 这样数据库密码等敏感信息就不用硬编码在代码里。
load_dotenv()

# --------------------------------------------------
# 2. 拼接数据库连接字符串
# --------------------------------------------------
# 格式：mysql+pymysql://用户名:密码@主机:端口/数据库名
DATABASE_URL = (
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', '3306')}/{os.getenv('DB_NAME')}"
)

# --------------------------------------------------
# 3. 创建数据库引擎（Engine）
# --------------------------------------------------
engine = create_engine(
    DATABASE_URL,
    echo=True,              # 打印执行的 SQL 语句到终端，方便调试。生产环境建议设为 False。
    pool_size=10,           # 连接池中保持的常驻连接数
    max_overflow=20,        # 超出 pool_size 后允许创建的最大额外连接数
    pool_timeout=30,        # 连接池满了之后，获取连接的最大等待时间（秒），超时抛异常
    pool_recycle=900,       # 连接存活的最大时间（秒），超过后自动回收重建
    pool_pre_ping=True,     # 每次从池中取连接时先发一个 ping 检测是否存活
)

# --------------------------------------------------
# 4. 自动建表
# --------------------------------------------------
def create_db_and_tables():
    SQLModel.metadata.create_all(engine)

# --------------------------------------------------
# 5. 提供 Session 的生成器函数（用于 FastAPI 依赖注入）
# --------------------------------------------------
def get_session():
    with Session(engine) as session:
        yield session

### 关键概念：Engine vs Session

| 概念 | 类比 | 作用 |
|---|---|---|
| **Engine** | 数据库的"总机" | 管理连接池，负责创建和回收连接。全局只创建一次。 |
| **Session** | 一次"通话" | 一次数据库交互的上下文。用完要关闭。每次 HTTP 请求创建一个。 |

# 第二步：定义数据模型

项目有四种模型，各司其职：

| 文件 | 用途 | 基类 | 是否建表 |
|---|---|---|---|
| `models/tb_user.py` | 数据库表模型（ORM） | `SQLModel, table=True` | 是 |
| `models/schemas.py` | 内部 DTO，repo→service 之间传递 | `BaseModel`（纯 Pydantic） | 否 |
| `models/request.py` | 请求体校验 | `BaseModel`（纯 Pydantic） | 否 |
| `models/response.py` | 统一响应格式 | `BaseModel`（纯 Pydantic） | 否 |

四者的关系和流转：
```
客户端发来 JSON 请求
    → Request 模型校验（字段类型对不对？必填字段有没有？）
    → Router 解析 Request 字段，传给 Service
    → Service 调用 Repository 操作数据库（ORM）
    → Repository 返回 Schema（DTO，如 UserPublic，不含敏感字段）
    → Service 将 Schema 转为 Response 模型
    → 返回 JSON 响应给客户端
```

In [ ]:
# app/models/tb_user.py — 数据库表模型
#
# SQLModel 同时继承了 SQLAlchemy 的 ORM 能力和 Pydantic 的数据校验能力。
# table=True 是关键：加上它，SQLModel 才会为这个类创建数据库表。

from sqlmodel import SQLModel, Field


class User(SQLModel, table=True):       # table=True → 会创建一张名为 "user" 的表
    # id 字段：自增主键
    id: int | None = Field(default=None, primary_key=True)
    # name 字段：str 类型，没有 default，创建时必须提供
    name: str
    # password 字段：存储的是 bcrypt 哈希后的密码，不是明文
    password: str
    # email 字段：数据库层强制唯一
    email: str = Field(unique=True)

In [ ]:
# app/models/schemas.py — 内部 DTO（Data Transfer Object）
#
# 这是 repo 层和 service 层之间传递数据的中间模型。
# 它的作用是：
#   1. 脱敏 — 不包含 password，service 层永远接触不到密码
#   2. 解耦 — service 层不依赖 User ORM 对象，只依赖纯数据模型
#   3. 明确边界 — repo 层返回 UserPublic，service 层的函数签名清晰可见

from pydantic import BaseModel


class UserPublic(BaseModel):
    """不含密码的用户公开模型，用于 repo -> service 层之间流转。"""
    id: int
    name: str
    email: str

In [ ]:
# app/models/request.py — 请求体模型
#
# 继承自 pydantic.BaseModel（不是 SQLModel），不会创建数据库表。
# FastAPI 会根据这些模型自动校验客户端发来的 JSON。

from pydantic import BaseModel


class UserCreateRequest(BaseModel):
    name: str
    password: str
    email: str


class UserLoginRequest(BaseModel):
    email: str
    password: str


class PasswordUpdateRequest(BaseModel):
    id: int
    old_password: str
    new_password: str


class UserUpdateRequest(BaseModel):
    id: int
    name: str

In [ ]:
# app/models/response.py — 响应体模型
#
# 统一 API 的返回格式，让前端知道响应的结构。
# 注意 UserResponse 只有 name 和 email，不暴露 password。

from typing import Any
from fastapi.responses import FileResponse
from pydantic import BaseModel


class APIResponse(BaseModel):
    code: int = 0                    # 默认 0（成功），非 0 表示各种错误
    message: str = "success"         # 默认 "success"
    data: Any = None                 # 可以是任意类型的数据


class UserResponse(BaseModel):
    name: str
    email: str


class PdfFileResponse(FileResponse):
    """PDF文件响应，用于返回下载生成的PDF文件。"""
    def __init__(self, path, filename=None):
        super().__init__(
            path,
            media_type="application/pdf",
            filename=filename,
        )

# 第三步：Repository 层 — `app/repositories/user.py`

Repository（仓库）层是数据访问的抽象，封装了所有 ORM 操作和密码哈希。

**核心原则：**
- 所有函数都以 `session: Session` 作为第一个参数
- 输入是基本类型（str、int），输出是 `UserPublic`（不含敏感字段的 DTO）或错误标识
- 密码哈希、验证等实现细节完全封装在 repo 内部，service 层无感
- 使用错误码（`"not_found"`、`"wrong_password"`）而非异常来传递可预期的业务错误

### Repo 层的转换角色

```
ORM 对象 (User，含 password)
    ↓ _to_public() 转换
DTO (UserPublic，不含 password)
    ↓ 返回给 Service
Service 只看到安全的数据
```

In [ ]:
# app/repositories/user.py — 数据访问层
#
# 这是新架构中抽出来的层，旧代码中这些逻辑全都在 services/user.py 里。
# 抽出来后，service 层变得非常薄，只做业务编排。

import bcrypt
from sqlmodel import Session, select

from app.models.tb_user import User
from app.models.schemas import UserPublic


# ── 内部转换工具（私有） ─────────────────────────────────────

def _to_public(user: User | None) -> UserPublic | None:
    """将 ORM 对象转为 DTO。如果传入 None 则返回 None。"""
    if user is None:
        return None
    return UserPublic(id=user.id, name=user.name, email=user.email)
#                       ↑ 注意：不包含 password 字段！


# ── 密码工具（私有，只在本模块内使用） ────────────────────────

def hash_password(password: str) -> str:
    """将明文密码哈希，返回哈希字符串。"""
    # bcrypt 需要 bytes 输入，所以 encode → hash → decode
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(plain_password: str, hashed_password: str) -> bool:
    """验证密码是否匹配。"""
    return bcrypt.checkpw(plain_password.encode("utf-8"), hashed_password.encode("utf-8"))


# ── 查询方法：返回 UserPublic ──────────────────────────────

def get_by_id(session: Session, user_id: int) -> UserPublic | None:
    """按主键查询。"""
    return _to_public(session.get(User, user_id))


def get_by_email(session: Session, email: str) -> UserPublic | None:
    """按邮箱查询。"""
    statement = select(User).where(User.email == email)
    return _to_public(session.exec(statement).first())


def get_by_name(session: Session, name: str) -> UserPublic | None:
    """按用户名查询。"""
    statement = select(User).where(User.name == name)
    return _to_public(session.exec(statement).first())


# ── 认证方法 ──────────────────────────────────────────────

def authenticate(session: Session, email: str, plain_password: str) -> tuple[UserPublic | None, str | None]:
    """
    验证用户登录凭据。
    注意：这里需要直接访问 User.password，所以不能先 _to_public。
    Returns:
        (UserPublic, None)         — 认证成功
        (None, "not_found")        — 用户不存在
        (None, "wrong_password")   — 密码错误
    """
    statement = select(User).where(User.email == email)
    user = session.exec(statement).first()
    if user is None:
        return None, "not_found"
    if not verify_password(plain_password, user.password):
        return None, "wrong_password"
    return _to_public(user), None    # 验证通过后才转为 UserPublic


def change_password(session: Session, user_id: int, old_plain: str, new_plain: str) -> str | None:
    """
    验证旧密码并更新为新密码。
    Returns: None（成功）| "not_found" | "wrong_password"
    """
    user = session.get(User, user_id)
    if user is None:
        return "not_found"
    if not verify_password(old_plain, user.password):
        return "wrong_password"
    user.password = hash_password(new_plain)
    session.add(user)
    session.commit()
    return None


# ── 写入方法 ──────────────────────────────────────────────

def create(session: Session, name: str, email: str, plain_password: str) -> UserPublic:
    """创建用户，内部处理密码哈希。"""
    user = User(
        name=name,
        password=hash_password(plain_password),  # 明文 → 哈希
        email=email,
    )
    session.add(user)
    session.commit()
    session.refresh(user)       # 拿到数据库生成的 id
    return _to_public(user)     # 返回 DTO，不含密码


def set_name(session: Session, user_id: int, new_name: str) -> UserPublic | None:
    """更新用户名。"""
    user = session.get(User, user_id)
    if user is None:
        return None
    user.name = new_name
    session.add(user)
    session.commit()
    session.refresh(user)
    return _to_public(user)


def remove(session: Session, user_id: int) -> bool:
    """删除用户，返回是否成功。"""
    user = session.get(User, user_id)
    if user is None:
        return False
    session.delete(user)
    session.commit()
    return True

### Repository 层的 ORM 操作速查

从上面的代码可以看到所有 CRUD 操作对应的 ORM 用法：

| 操作 | ORM 代码 | 对应 SQL |
|---|---|---|
| **主键查询** | `session.get(User, user_id)` | `SELECT * FROM user WHERE id = ?` |
| **条件查询** | `session.exec(select(User).where(User.email == email)).first()` | `SELECT * FROM user WHERE email = ? LIMIT 1` |
| **新增** | `session.add(user)` + `session.commit()` + `session.refresh(user)` | `INSERT INTO user ...` |
| **更新** | 修改属性 → `session.add(user)` + `session.commit()` | `UPDATE user SET ... WHERE id = ?` |
| **删除** | `session.delete(user)` + `session.commit()` | `DELETE FROM user WHERE id = ?` |

### 常见陷阱

| 陷阱 | 说明 |
|---|---|
| 忘记 `commit()` | `add` / `delete` 只是暂存，不 `commit` 不会真正写入数据库 |
| 忘记 `refresh()` | 新增后如果不 `refresh`，对象的 `id` 仍然是 `None` |
| 修改属性后忘记 `add` | 直接改属性后必须 `session.add(obj)` 再 `commit` |
| 查询结果为 None | `session.get()` 和 `.first()` 可能返回 `None`，使用前一定要判空 |

# 第四步：Service 层 — `app/services/user.py`

抽出 repo 层后，service 变得非常薄，**只做业务编排**：
- 调用 repo 层查询/修改数据
- 根据返回结果决定错误码和消息
- 将 `UserPublic`（DTO）转为 `UserResponse`（响应模型）

**Service 层不再：**
- 直接操作 ORM（session.get、select 等）
- 接触密码哈希（bcrypt）
- 接触 `User` ORM 对象（只接触 `UserPublic`）

In [ ]:
# app/services/user.py — 业务逻辑层
#
# 对比旧版本：
#   旧版 service 直接用 session.get()、select() 操作数据库，还包含密码哈希工具。
#   新版 service 只调用 repo 层的方法，职责非常清晰。

from sqlmodel import Session

from app.models.response import APIResponse, UserResponse
from app.models.schemas import UserPublic
from app.repositories import user as user_repo


# ── DTO → Response 转换 ─────────────────────────────────────

def _to_response(user: UserPublic) -> UserResponse:
    """将内部 DTO (UserPublic) 转为对外响应 (UserResponse)。
    UserPublic 有 id/name/email，UserResponse 只有 name/email。"""
    return UserResponse(name=user.name, email=user.email)


# ── 业务逻辑 ──────────────────────────────────────────────

def register(session: Session, name: str, password: str, email: str) -> APIResponse:
    # 业务规则：邮箱不能重复
    existing = user_repo.get_by_email(session, email)
    if existing is not None:
        return APIResponse(code=1, message="email already registered")
    # 调用 repo 创建用户（密码哈希在 repo 内部处理）
    public = user_repo.create(session, name, email, password)
    return APIResponse(data=_to_response(public))


def login(session: Session, email: str, password: str) -> APIResponse:
    # 调用 repo 认证（密码验证在 repo 内部处理）
    public, error = user_repo.authenticate(session, email, password)
    if error == "not_found":
        return APIResponse(code=1, message="user not found")
    if error == "wrong_password":
        return APIResponse(code=2, message="wrong password")
    return APIResponse(data=_to_response(public))


def update_password(session: Session, user_id: int, old_password: str, new_password: str) -> APIResponse:
    error = user_repo.change_password(session, user_id, old_password, new_password)
    if error == "not_found":
        return APIResponse(code=1919810, message="user not found")
    if error == "wrong_password":
        return APIResponse(code=2, message="wrong password")
    return APIResponse(message="password updated")


def update_name(session: Session, user_id: int, name: str) -> APIResponse:
    public = user_repo.get_by_id(session, user_id)
    if public is None:
        return APIResponse(code=1919810, message="user not found")
    existing = user_repo.get_by_name(session, name)
    if existing is not None:
        return APIResponse(code=1, message="username already exists")
    public = user_repo.set_name(session, user_id, name)
    return APIResponse(data=_to_response(public))


def delete_user(session: Session, user_id: int) -> APIResponse:
    if not user_repo.remove(session, user_id):
        return APIResponse(code=1919810, message="user not found")
    return APIResponse(message="user deleted")


def read_user(session: Session, user_id: int) -> APIResponse:
    public = user_repo.get_by_id(session, user_id)
    if public is None:
        return APIResponse(code=1919810, message="user not found")
    return APIResponse(data=_to_response(public))

### Service 层的模式总结

观察上面每个函数，它们都遵循同一个模式：

```
1. 调用 repo 方法（可能先查，再改）
2. 检查返回结果（是否 None？是否有 error？）
3. 根据结果返回 APIResponse（成功/失败）
```

这种模式的好处：
- 业务逻辑一目了然，没有 ORM 细节干扰
- 错误处理统一用 APIResponse 的 code + message
- 测试时只需 mock repo 层的方法，不用 mock 数据库

# 第五步：Router 层 — `app/routers/user.py`

路由函数只做三件事：**接收 HTTP 请求 → 调用 service → 返回 HTTP 响应**。

关键机制是 `Depends(get_session)`：
- FastAPI 在处理请求前自动调用 `get_session()` 创建一个数据库 session
- 把 session 作为参数传给路由函数
- 请求处理完后自动关闭 session

In [ ]:
# app/routers/user.py

from fastapi import APIRouter, Depends, status
from sqlmodel import Session

from app.core import get_session
from app.models.response import APIResponse
from app.models.request import UserCreateRequest, UserLoginRequest, PasswordUpdateRequest, UserUpdateRequest
from app.services import user as user_service

# APIRouter 用于把路由分组。
# prefix="/user" 表示这个文件里所有路由都以 /user 开头。
# tags=["user"] 用于 /docs 页面的分组显示。
router = APIRouter(prefix="/user", tags=["user"])


@router.get("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def read_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.read_user(session, user_id)


@router.post("/register", status_code=status.HTTP_201_CREATED, response_model=APIResponse)
def register(req: UserCreateRequest, session: Session = Depends(get_session)):
    # req 是请求体，FastAPI 自动解析 JSON 并用 UserCreateRequest 校验
    return user_service.register(session, req.name, req.password, req.email)


@router.post("/login", status_code=status.HTTP_200_OK, response_model=APIResponse)
def login(req: UserLoginRequest, session: Session = Depends(get_session)):
    return user_service.login(session, req.email, req.password)


@router.put("/password", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_password(req: PasswordUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_password(session, req.id, req.old_password, req.new_password)


@router.put("/name", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_name(req: UserUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_name(session, req.id, req.name)


@router.delete("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def delete_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.delete_user(session, user_id)

# 第六步：App 入口 — `app/main.py`

这是整个应用的入口，做三件事：
1. **lifespan** — 应用启动时调用 `create_db_and_tables()` 自动建表
2. 创建 FastAPI 实例
3. 挂载所有路由（user_router + comic_router）

In [ ]:
# app/main.py

from contextlib import asynccontextmanager

from fastapi import FastAPI, status

from app.core import create_db_and_tables
from app.models.response import APIResponse
from app.routers import user as user_router
from app.routers import comic as jmcomic_router


@asynccontextmanager
async def lifespan(app):
    create_db_and_tables()   # 启动时：自动建表
    yield                    # 应用运行中...


app = FastAPI(lifespan=lifespan)

app.include_router(user_router.router)
app.include_router(jmcomic_router.router)


@app.get("/", status_code=status.HTTP_200_OK)
def read_root():
    return APIResponse(
        data={"content": "Hello, world!"},
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

# 整体请求流程

以注册用户为例，一个完整的 HTTP 请求走过的路径：

```
1. 客户端发送 POST /user/register  {"name": "张三", "password": "123456", "email": "zs@example.com"}

2. FastAPI 接收请求
   ├── 路由匹配 → routers/user.py 的 register 函数
   ├── 请求体校验 → UserCreateRequest 自动校验字段类型和必填
   └── 依赖注入 → Depends(get_session) 自动创建数据库 session

3. Router 调用 Service
   → user_service.register(session, "张三", "123456", "zs@example.com")

4. Service 调用 Repository（业务编排）
   ├── user_repo.get_by_email(session, "zs@example.com")  → 检查邮箱是否重复
   └─│ user_repo.create(session, "张三", "zs@example.com", "123456")
       │   ├── User(name="张三", password=hash("123456"), email="zs@example.com")
       │   ├── session.add(user)   → 加入暂存区
       │   ├── session.commit()    → 执行 INSERT INTO user ...
       │   └── session.refresh()   → 拿到数据库生成的 id
       └── 返回 UserPublic(id=1, name="张三", email="zs@example.com")  ← DTO，不含密码

5. Service 将 DTO 转为 Response
   → UserPublic → UserResponse(name="张三", email="zs@example.com")
   → 包装成 APIResponse(code=0, message="success", data={...})

6. Router 把 APIResponse 转成 JSON 返回给客户端
   → {"code": 0, "message": "success", "data": {"name": "张三", "email": "zs@example.com"}}

7. 请求结束，session 自动关闭，连接归还给连接池
```

### 数据在各层间的转换链

```
Request (Pydantic)  →  基本类型参数  →  [Repo]  →  UserPublic (DTO)  →  [Service]  →  UserResponse (Pydantic)
   ↑ 客户端传入          ↑ Router 解析      ↑ ORM 操作       ↑ 脱敏          ↑ 业务判断      ↑ 返回客户端
```